In [1]:
import os
import pandas as pd
import shutil
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import numpy as np
import re


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\XuanR\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
def roughmatch(df1,df2):
    # match based on the column "word"

    # df1 = df_segment2word.copy()
    # df2 = df_gentle.copy()
    df1['word_index'] = df1.groupby('word').cumcount()
    df2['word_index'] = df2.groupby('word').cumcount()

    # Step 2: Merge using both 'word' and 'word_index'
    merged_df = pd.merge(df1, df2, on=['word', 'word_index'], how='left')

    # Step 3: Drop the helper column if no longer needed
    merged_df = merged_df.drop(columns=['word_index'])

    # Step 4: check if the onset of the next word is earlier than the previous one
    merged_df.loc[1:,'flag_onset'] = merged_df.loc[1:,'onset'].values - merged_df.loc[0:merged_df.shape[0]-2,'onset'].values
    merged_df.loc[merged_df['flag_onset']<0,'onset'] = np.nan

    merged_df['duration'] = merged_df['offset']-merged_df['onset']
    merged_df.loc[merged_df['duration']<=0,'offset'] = np.nan

    # Step 4: replace the missing onset with the offset of the last word, the missing offset with the onset of the next word
    List_idx = merged_df.loc[merged_df['onset'].isna()].index
    List_idx = [x for x in List_idx.to_list() if 0 not in List_idx.tolist()]
    List_idx2 = [x-1 for x in List_idx]
    merged_df.loc[List_idx,'onset'] = merged_df.loc[List_idx2,'offset'].values

    List_idx = merged_df.loc[merged_df['offset'].isna()].index
    List_idx = [x for x in List_idx.to_list() if merged_df.shape[0]-1 not in List_idx.tolist()]
    List_idx2 = [x+1 for x in List_idx]
    merged_df.loc[List_idx,'offset'] = merged_df.loc[List_idx2,'onset'].values

    return(merged_df)

In [4]:
Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github'
Dir_segments = os.path.join(Dir_github,'derivatives','behavioral','segmentation')
Dir_gentle = os.path.join(Dir_github,'derivatives','behavioral','timestamps','timestamps_word')
repo = 'ABC_valence'

# Unzip

I used a Mac computer to extract the timestamps for each word manually using Gentle. It generates a list of zip files. 

In [ ]:
# skip if this step has already been done.
List_files_zip = [x for x in os.listdir(os.path.join(Dir_github,'data','timestamps','zips')) if '.zip' in x]
for filename in List_files_zip:    
    Dir_output = os.path.join(Dir_github,'derivatives','behavioral','timestamps','timestamps_word',filename.replace('.zip',''))
    shutil.unpack_archive(os.path.join(Dir_github,'derivatives','behavioral','timestamps','zips',filename), Dir_output)

In [ ]:
List_audios = [x for x in os.listdir(Dir_gentle)]
for audio in List_audios:
    df_gentle = pd.read_csv(os.path.join(Dir_gentle,audio,'align.csv'),header=None,)
    df_gentle.columns = ['word','word_gentle','onset','offset']
    df_gentle['word'] = df_gentle['word'].str.replace("’","'")
    if not os.path.exists(os.path.join(Dir_gentle,audio,'align_RepApos.csv')):
        df_gentle.to_csv(os.path.join(Dir_gentle,audio,'align_RepApos.csv'),index=False)

# Segment level

Get the onset and offset timestamps for each segment.

In [22]:
List_files_seg = [x for x in os.listdir(Dir_segments) if 'csv' in x]
List_files_gentle = [x for x in os.listdir(Dir_gentle)]

separators = [',', ':', ';', ' ','.','-','!','?','"','“','”','--','/','(',')']
pattern = f"[{''.join(re.escape(s) for s in separators)}]"

df_punc = pd.DataFrame({'wrong':['”','“',"‘","’","–"],
                       'correct':['"','"',"'","'","-"]})

Dir_output = os.path.join(Dir_github,'derivatives','behavioral','timestamps','timestamps_segment')
if not os.path.exists(Dir_output):
    os.mkdir(Dir_output)

dfs_segment2word = []
dfs_segment = []
for file in List_files_seg:
    _,story,_ = file.split('_')
    List_audios = [x for x in List_files_gentle if story in x]
    for audio in List_audios: # loop through two voices
        filename = f"timestamps_{audio}.csv"
        if os.path.exists(os.path.join(Dir_output,filename)):
            df_segment = pd.read_csv(os.path.join(Dir_output,filename))
            df_segment2word = pd.read_csv(os.path.join(Dir_output,f"timestamps_{audio}_segment2word.csv"))
            dfs_segment.append(df_segment)
            dfs_segment2word.append(df_segment2word)
            continue
        
        # import transcript
        df_segment = pd.read_csv(os.path.join(Dir_segments,f'segment_{story}_formatted.csv'))
        df_segment['audio'] = audio
        for i,row in df_punc.iterrows():
                
            df_segment['segment'] = df_segment['segment'].str.replace(row['wrong'],row['correct'])
            df_segment['reason'] = df_segment['reason'].str.replace(row['wrong'],row['correct'])
        
        df_segment['segment_raw'] = df_segment['segment']


        # also export the word level timestamps
        df_segment2word = []
        for i, row in df_segment.iterrows():
            tokens = re.split(pattern, row['segment'])
            words = [token for token in tokens if token]
            df_segment2word.append(pd.DataFrame({'word':words,'idx_segment':row['idx'],'segment':row['segment'],'segment_raw':row['segment_raw'],'reason':row['reason'],
                                                 'story':row['story'],'audio':row['audio']}))
        df_segment2word = pd.concat(df_segment2word,ignore_index=True)

        # import gentle generated timestamps
        # df_gentle = pd.read_csv(os.path.join(Dir_gentle,audio,'align_RepApos.csv'))
        df_gentle = pd.read_csv(os.path.join(Dir_gentle,audio,'align.csv'),header=None,)
        df_gentle.columns = ['word','word_gentle','onset','offset']
        # extract onset and offset timestamps
        df_segment2word = roughmatch(df_segment2word,df_gentle)

        for i,row in df_segment.iterrows():
            df_tmp = df_segment2word.loc[df_segment2word['idx_segment']==i].reset_index(drop=True)
            df_segment.loc[i,'onset'] = df_tmp.loc[0,'onset']
            df_segment.loc[i,'offset'] = df_tmp.loc[df_tmp.shape[0]-1,'offset']
            df_segment.loc[i,'words'] = '|'.join(df_tmp['word'].values.tolist())

        filename = f"timestamps_{audio}.csv"
        df_segment.to_csv(os.path.join(Dir_output,filename),index=False)
        dfs_segment.append(df_segment)

        filename = f"timestamps_{audio}_segment2word.csv"
        df_segment2word.to_csv(os.path.join(Dir_output,filename),index=False)
        dfs_segment2word.append(df_segment2word)

dfs_segment2word = pd.concat(dfs_segment2word,ignore_index=True)        
dfs_segment = pd.concat(dfs_segment,ignore_index=True)   

Summary

In [23]:
df_summary_byStory = dfs_segment[['story','audio','segment']].groupby(['story','audio']).count().reset_index()
df_summary_byStory = df_summary_byStory.iloc[::2]  # even
SegsPerStory_mean = df_summary_byStory['segment'].mean().round(2)
SegsPerStory_std = round(df_summary_byStory['segment'].std(),2)
print(f"On average, a story has {SegsPerStory_mean} (SD = {SegsPerStory_std}) segments")
df_summary_byStory

df_summary_bySeg = dfs_segment2word[['story','idx_segment','word']].groupby(['story','idx_segment']).count().reset_index()
WordsPerSeg_mean = df_summary_bySeg['word'].mean().round(2)
WordsPerSeg_std = round(df_summary_bySeg['word'].std(),2)
print(f"On average across stoires, a segment has {WordsPerSeg_mean} (SD = {WordsPerSeg_std}) words")
df_summary_bySeg.groupby('story').mean().reset_index()


On average, a story has 130.43 (SD = 21.77) segments
On average across stoires, a segment has 13.57 (SD = 6.98) words


,story,idx_segment,word
0,AgedMother,63.5,14.000000
1,Aqua,73.0,13.523810
2,BiggestWastes,75.5,12.118421
3,BradfordBoar,78.0,13.859873
4,BulliesPronoun,78.5,12.759494
5,BulliesRepeatedNames,73.5,12.878378
6,Elvis,78.0,12.993631
7,EmotionSentences,41.5,8.452381
8,EmpireState,56.0,17.309735
9,FairyTree,54.5,14.709091


In [25]:
dfs_segment2word = []
dfs_segment = []
for file in List_files_seg:
    _,story,_ = file.split('_')
    List_audios = [x for x in List_files_gentle if story in x]
    for audio in List_audios: # loop through two voices
        filename = f"timestamps_{audio}_segment2word.csv"
        df = pd.read_csv(os.path.join(Dir_output,filename))
        dfs_segment2word.append(df)

        filename = f"timestamps_{audio}_segment2word.csv"
        df = pd.read_csv(os.path.join(Dir_output,filename))
        dfs_segment.append(df)
dfs_segment2word = pd.concat(dfs_segment2word,ignore_index=True)
dfs_segment2word

,word,idx_segment,segment,segment_raw,reason,story,audio,word_gentle,onset,offset,flag_onset,duration
0,Long,0,"Long, long ago","Long, long ago",positioning element,AgedMother,Allison_AgedMother,long,4.48,4.89,NaN,0.41
1,long,0,"Long, long ago","Long, long ago",positioning element,AgedMother,Allison_AgedMother,long,4.90,5.18,0.42,0.28
2,ago,0,"Long, long ago","Long, long ago",positioning element,AgedMother,Allison_AgedMother,ago,5.20,5.71,0.30,0.51
3,there,1,there lived at the foot of the mountain,there lived at the foot of the mountain,positioning element,AgedMother,Allison_AgedMother,there,5.86,6.06,0.66,0.20
4,lived,1,there lived at the foot of the mountain,there lived at the foot of the mountain,positioning element,AgedMother,Allison_AgedMother,lived,6.06,6.38,0.20,0.32
...,...,...,...,...,...,...,...,...,...,...,...,...
40699,that,137,that everyone will want one.,that everyone will want one.,relative clause; >maximal segment,UserFriendly,Jacob_UserFriendly,NaN,NaN,NaN,NaN,NaN
40700,everyone,137,that everyone will want one.,that everyone will want one.,relative clause; >maximal segment,UserFriendly,Jacob_UserFriendly,NaN,NaN,NaN,NaN,NaN
40701,will,137,that everyone will want one.,that everyone will want one.,relative clause; >maximal segment,UserFriendly,Jacob_UserFriendly,NaN,NaN,NaN,NaN,NaN
40702,want,137,that everyone will want one.,that everyone will want one.,relative clause; >maximal segment,UserFriendly,Jacob_UserFriendly,NaN,NaN,NaN,NaN,NaN
